# 🎭 FreeFakeStudio — 1-Click Web Edition
**Zero-setup GPU Engine for the FreeFakeStudio Web App**

1. Select your default model below.
2. Click the **▶️ Play** button on the left.
3. Once ready, click the **🚀 Open FreeFakeStudio Web App** button that appears!

In [ ]:
# @title 🚀 1-Click Launch FreeFakeStudio GPU Engine
# @markdown ### 👇 Select default model & click the ▶️ Play button
Model = "⚡ Z-Image Turbo (Fastest ~1 min)"  # @param ["⚡ Z-Image Turbo (Fastest ~1 min)", "🌊 FLUX.2-klein 4B", "🔮 FLUX.2-klein 9B", "🎨 Qwen-Image-Edit", "🖌️ ERNIE-Image Turbo"]

import os, sys, subprocess, time, re, glob
import torch
from IPython.display import display, HTML, clear_output

# ── 1. Check GPU ──────────────────────────────────────────
if not torch.cuda.is_available():
    print("\n" + "="*60)
    print("⚠️ GPU NOT DETECTED!")
    print("Please enable Google's free T4 GPU:")
    print("1. Click 'Runtime' in the top menu")
    print("2. Click 'Change runtime type'")
    print("3. Choose 'T4 GPU' -> Save")
    print("4. Run this cell again")
    print("="*60 + "\n")
    raise SystemExit("GPU required to run FreeFakeStudio")

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU Connected: {gpu_name}")
print("⏳ Step 1/3: Setting up ComfyUI backend & dependencies...")

# ── 2. ComfyUI & Deps ─────────────────────────────────────
need_gguf = any(k in Model for k in ["Qwen", "9B", "ERNIE"])
need_diffusers = "Qwen" in Model

if not os.path.exists('/content/ComfyUI'):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", "/content/ComfyUI"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/requirements.txt"], check=True)

if need_gguf and not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    subprocess.run(["git", "clone", "https://github.com/city96/ComfyUI-GGUF.git", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt"], check=True)

if need_diffusers:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/diffusers"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg[gpu]", "onnxruntime-gpu", "aria2", "gradio"], check=True)

# ── 3. Model Dirs & Downloads ─────────────────────────────
DIFF   = '/content/ComfyUI/models/diffusion_models'
CLIP   = '/content/ComfyUI/models/clip'
TXTENC = '/content/ComfyUI/models/text_encoders'
VAE    = '/content/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if os.path.exists(path) and os.path.getsize(path) > 1024:
        print(f'  ✅ {name} (cached)')
    else:
        if os.path.exists(path):
            os.remove(path)
        print(f'  ⏳ Downloading {name}...')
        subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", dest_dir, "-o", name], check=True)
        print(f'  ✅ {name} downloaded!')

print(f"\n📦 Step 2/3: Downloading model for {Model}...")

if "Z-Image Turbo" in Model:
    dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

elif "FLUX.2-klein 4B" in Model:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b.safetensors', DIFF, 'flux-2-klein-4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors', TXTENC, 'qwen_3_4b_fp4_flux2.safetensors')

elif "FLUX.2-klein 9B" in Model:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors', DIFF, 'flux-2-klein-9b-kv-fp8.safetensors')
    dl('https://huggingface.co/unsloth/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q2_K_L.gguf', TXTENC, 'Qwen3-8B-Q2_K_L.gguf')

elif "Qwen-Image-Edit" in Model:
    dl('https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q3_K_M.gguf', DIFF, 'qwen-image-edit-2511-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/mmproj-F16.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf')
    dl('https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

elif "ERNIE-Image Turbo" in Model:
    dl('https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/resolve/main/ernie-image-turbo-Q6_K.gguf', DIFF, 'ernie-image-turbo-Q6_K.gguf')
    dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/text_encoders/ministral-3-3b.safetensors', TXTENC, 'ministral-3-3b.safetensors')
    dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

# Symlink clip/ <-> text_encoders/
for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)
for f in glob.glob(f'{TXTENC}/*'):
    link = os.path.join(CLIP, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)

# ── 4. Clone / Pull FreeFakeStudio ────────────────────────
print("\n🚀 Step 3/3: Launching FreeFakeStudio Engine...")
REPO = 'https://github.com/MuntasirMalek/FreeFakeStudio.git'
if os.path.exists('/content/FreeFakeStudio'):
    os.chdir('/content/FreeFakeStudio')
    subprocess.run(["git", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO, "/content/FreeFakeStudio"], check=True)
    os.chdir('/content/FreeFakeStudio')

# ── 5. Run app.py and capture live link ────────────────────
proc = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

gradio_url = None
url_pattern = re.compile(r'(https://[a-zA-Z0-9-]+\.gradio\.live)')

for line in proc.stdout:
    print(line, end="")
    match = url_pattern.search(line)
    if match:
        gradio_url = match.group(1)
        break

if gradio_url:
    web_app_url = f"https://muntasirmalek.github.io/FreeFakeStudio/?backend={gradio_url}"
    clear_output()
    display(HTML(f'''
    <div style="background: linear-gradient(135deg, #090d16, #111827); border: 2px solid #3b82f6; border-radius: 18px; padding: 32px; text-align: center; max-width: 680px; margin: 25px auto; box-shadow: 0 20px 45px rgba(0,0,0,0.6); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <div style="font-size: 48px; margin-bottom: 12px;">🎉</div>
        <h2 style="color: #60a5fa; margin: 0 0 10px 0; font-size: 26px; font-weight: 800; letter-spacing: -0.02em;">
            FreeFakeStudio GPU Engine is LIVE!
        </h2>
        <p style="color: #cbd5e1; font-size: 15px; margin: 0 0 24px 0; line-height: 1.6;">
            Keep this Colab tab open in the background.<br>
            Click the button below to start creating on your website:
        </p>
        <a href="{web_app_url}" target="_blank" style="display: inline-block; background: linear-gradient(135deg, #2563eb, #7c3aed); color: #ffffff; padding: 16px 36px; border-radius: 12px; font-size: 18px; font-weight: 700; text-decoration: none; box-shadow: 0 6px 20px rgba(37,99,235,0.45); transition: transform 0.2s;">
            🚀 Open FreeFakeStudio Web App
        </a>
        <div style="margin-top: 24px; padding-top: 18px; border-top: 1px solid #1f2937; font-size: 13px; color: #94a3b8;">
            Public GPU Tunnel: <a href="{gradio_url}" target="_blank" style="color: #38bdf8; text-decoration: none;">{gradio_url}</a>
        </div>
    </div>
    '''))

# Keep process running
try:
    for line in proc.stdout:
        pass
except KeyboardInterrupt:
    proc.terminate()
    print("\n🛑 Engine stopped.")
